# 🔥 Hệ thống Phát hiện Cháy Sớm — Training Notebook
### Early Fire Detection System — RT-DETR + SAHI

**Tác giả:** Vo Xuan Quang (523H0173) & Hoang Xuan Thanh (523H0178)  
**Trường:** Đại học Tôn Đức Thắng (TDTU)

---

## Notebook này hướng dẫn:
1. ✅ Cài đặt dependencies
2. 📊 Khám phá dataset (EDA)
3. 🏋️ Training 3 stages
4. 📈 Đánh giá model
5. 🔍 Inference (test)
6. 🚀 SAHI inference cho vật thể nhỏ

## 1. ✅ Cài đặt Dependencies

Chạy cell bên dưới để cài đặt tất cả thư viện cần thiết.

In [ ]:
# Cài đặt dependencies
!pip install -q -r requirements.txt

# Kiểm tra GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("Apple MPS (Metal) available")
else:
    print("⚠️ Không có GPU — Training sẽ chậm!")

## 2. 📊 Khám phá Dataset (EDA)

Trước khi training, cần kiểm tra:
- Số lượng ảnh mỗi folder
- Phân bố classes (Fire vs Smoke)
- Kích thước ảnh
- Mẫu ảnh representative

In [ ]:
# Load config
from src.config import load_config
config = load_config('configs/default.yaml')
print(f"Model: {config.model.architecture}")
print(f"Image size: {config.model.img_size}")
print(f"Classes: {list(config.data.class_names)}")

In [ ]:
# Thống kê dataset
from src.data.dataset import FireSmokeDataset

all_folders = [
    '01_Positive_Standard',
    '02_Alley_Context', 
    '03_Negative_Hard_Samples',
    '04_SAHI_Small_Objects',
    '05_Real_Situation',
]

dataset = FireSmokeDataset(config, all_folders)
stats = dataset.get_stats()

print(f"\n📊 THỐNG KÊ DATASET:")
print(f"{'='*50}")
print(f"Tổng số ảnh: {stats['total_images']}")
print(f"Tổng số labels: {stats['total_labels']}")
print(f"\nPhân bố classes:")
for cls, count in stats['class_counts'].items():
    print(f"  {cls}: {count} objects")
print(f"\nChi tiết từng folder:")
for folder, info in stats['folder_stats'].items():
    print(f"  {folder}: {info['images']} ảnh, {info['objects']} objects")

In [ ]:
# Validate labels
from src.data.preprocessing import validate_yolo_labels, check_image_quality

print("\n🔍 KIỂM TRA LABELS:")
for folder in all_folders:
    print(f"\n--- {folder} ---")
    validate_yolo_labels(f'data/{folder}/labels', num_classes=2)

In [ ]:
# Visualize mẫu ảnh
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

def show_sample_images(folder_name, num_samples=4):
    """Hiển thị mẫu ảnh từ 1 folder."""
    images_dir = Path(f'data/{folder_name}/images')
    if not images_dir.exists():
        print(f"⚠️ Không tìm thấy: {images_dir}")
        return
    
    img_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))
    if not img_files:
        print(f"⚠️ Không có ảnh trong {folder_name}")
        return
    
    num_samples = min(num_samples, len(img_files))
    fig, axes = plt.subplots(1, num_samples, figsize=(4*num_samples, 4))
    if num_samples == 1:
        axes = [axes]
    
    for i, img_file in enumerate(img_files[:num_samples]):
        img = cv2.imread(str(img_file))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img)
        axes[i].set_title(img_file.name, fontsize=8)
        axes[i].axis('off')
    
    fig.suptitle(folder_name, fontsize=14)
    plt.tight_layout()
    plt.show()

# Hiển thị mẫu từ mỗi folder
for folder in all_folders:
    show_sample_images(folder)

## 3. 🏋️ Training — 3 Stage Pipeline

### Pipeline overview:

| Stage | Data | Mục tiêu | LR |
|---|---|---|---|
| 1. Baseline | 01 + 02 | Nhận biết lửa/khói cơ bản | 1e-4 |
| 2. Hard Negative | 01 + 02 + 03 | Giảm false positive | 5e-5 |
| 3. SAHI | 04 + 05 | Phát hiện vật thể nhỏ | 2e-5 |

Mỗi stage kế thừa weights từ stage trước → **Transfer learning** dần dần.

In [ ]:
# Khởi tạo model và trainer
from src.models.rtdetr_model import FireDetectionModel
from src.engine.trainer import Trainer

model = FireDetectionModel(config)
trainer = Trainer(model, config)

In [ ]:
# ============================================================
# 🥇 STAGE 1: BASELINE TRAINING
# ============================================================
# Data: 01_Positive_Standard + 02_Alley_Context
# Mục tiêu: Model học nhận biết lửa/khói cơ bản
# ============================================================

stage1_weights = trainer.run_baseline_training()
print(f"\n✅ Stage 1 weights: {stage1_weights}")

In [ ]:
# ============================================================
# 🥈 STAGE 2: HARD NEGATIVE MINING
# ============================================================
# Data: thêm 03_Negative_Hard_Samples
# Mục tiêu: Giảm false positive (hơi phở, đèn đỏ, etc.)
# ============================================================

stage2_weights = trainer.run_hard_negative_mining()
print(f"\n✅ Stage 2 weights: {stage2_weights}")

In [ ]:
# ============================================================
# 🥉 STAGE 3: SAHI FINE-TUNING
# ============================================================
# Data: 04_SAHI_Small_Objects + 05_Real_Situation
# Mục tiêu: Phát hiện vật thể nhỏ/xa
# ============================================================

stage3_weights = trainer.run_sahi_finetuning()
print(f"\n✅ FINAL weights: {stage3_weights}")

## 4. 📈 Đánh giá Model

Đánh giá model cuối cùng trên validation set.

In [ ]:
# Đánh giá model
from src.engine.evaluator import Evaluator

# Load model tốt nhất
best_model = FireDetectionModel(config, weights_path=stage3_weights)
evaluator = Evaluator(best_model, config)

# Chuẩn bị dataset đánh giá (dùng tất cả data)
eval_dataset = FireSmokeDataset(config, all_folders)
data_yaml = eval_dataset.prepare()

# Chạy evaluation
metrics = evaluator.evaluate(data_yaml)

# Lưu báo cáo
evaluator.save_report(metrics)

In [ ]:
# Vẽ training curves
from src.utils.visualization import plot_training_curves

# Vẽ curves cho từng stage
for stage_name in ['stage1_baseline', 'stage2_hard_negative', 'stage3_sahi']:
    log_dir = f'runs/train/{stage_name}'
    if Path(log_dir).exists():
        plot_training_curves(log_dir)

In [ ]:
# Benchmark FPS
# Cần 1 ảnh test để đo tốc độ
import glob
test_images = glob.glob('data/*/images/*.jpg')[:1]
if test_images:
    benchmark = best_model.benchmark(test_images[0], num_runs=50)
    print(f"\n📊 Benchmark:")
    print(f"   FPS: {benchmark['fps']}")
    print(f"   Avg latency: {benchmark['avg_ms']}ms")
else:
    print("⚠️ Không tìm thấy ảnh test")

## 5. 🔍 Inference — Test trên ảnh mới

In [ ]:
# Inference trên 1 ảnh
from src.utils.visualization import draw_detections_fancy

# Chọn ảnh test
test_image_path = 'data/01_Positive_Standard/images/'  # ← Thay bằng ảnh cụ thể
test_images = glob.glob(f'{test_image_path}*.jpg') + glob.glob(f'{test_image_path}*.png')

if test_images:
    img_path = test_images[0]
    print(f"Testing: {img_path}")
    
    # Detect
    detections = best_model.predict(img_path)
    
    # In kết quả
    print(f"\nPhát hiện {len(detections)} vật thể:")
    for det in detections:
        print(f"  {det['class_name']}: {det['confidence']:.2%} at {[int(x) for x in det['bbox']]}")
    
    # Vẽ bbox
    img = cv2.imread(img_path)
    result = draw_detections_fancy(img, detections)
    
    # Hiển thị
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.title(f'Detection Result — {len(detections)} objects found')
    plt.axis('off')
    plt.show()
else:
    print("⚠️ Thêm ảnh vào data/ để test")

## 6. 🔍 SAHI Inference — Phát hiện vật thể nhỏ

SAHI chia ảnh lớn thành patches nhỏ → phát hiện vật thể nhỏ/xa tốt hơn.

In [ ]:
# SAHI Inference
if test_images:
    img_path = test_images[0]
    print(f"SAHI Testing: {img_path}")
    
    # Detect với SAHI
    sahi_detections = best_model.predict_with_sahi(img_path)
    
    # So sánh
    normal_detections = best_model.predict(img_path)
    print(f"\n📊 So sánh:")
    print(f"  Normal detect: {len(normal_detections)} objects")
    print(f"  SAHI detect:   {len(sahi_detections)} objects")
    
    # Vẽ kết quả SAHI
    img = cv2.imread(img_path)
    result = draw_detections_fancy(img, sahi_detections)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.title(f'SAHI Detection — {len(sahi_detections)} objects')
    plt.axis('off')
    plt.show()
else:
    print("⚠️ Thêm ảnh vào data/ để test")

## 🎉 Hoàn thành!

### Bước tiếp theo:
1. Export model: `model.export(format='onnx')`
2. Chạy web server: `uvicorn web.main:app --port 8000`
3. Deploy lên Docker: `docker build -t fire-detection .`

---
*Vo Xuan Quang & Hoang Xuan Thanh — TDTU 2026*